In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path

def load_quadrat_layers(root, quadrat_id, year,
                        cover_path="Data/Quadrat_Spatial_Data/Shapefiles/Cover_All.shp",
                        density_path="Data/Quadrat_Spatial_Data/Shapefiles/Density_All.shp",
                        quad_field="QUAD_ID", year_field="YEAR"):
    """
    Returns (cover_gdf_filtered, density_gdf_filtered) for the requested quadrat/year.
    """
    root = Path(root)
    cover = gpd.read_file(root/cover_path)
    density = gpd.read_file(root/density_path)

    # If your field names differ, change quad_field/year_field above or detect here:
    for gdf in (cover, density):
        cols = {c.lower(): c for c in gdf.columns}
        if quad_field not in gdf.columns:
            quad_field = cols.get("quadrat") or cols.get("quadrat_id") or cols.get("quad_id") or list(gdf.columns)[0]
        if year_field not in gdf.columns:
            year_field = cols.get("year") or list(gdf.columns)[0]

    csel = cover[(cover[quad_field]==quadrat_id) & (cover[year_field]==year)]
    dsel = density[(density[quad_field]==quadrat_id) & (density[year_field]==year)]
    return csel, dsel

def plot_quadrat(cover_gdf, density_gdf, title=None, label_field="SCI_NAME",
                 figsize=(5,5), dpi=200, out_png=None):
    """
    Plots a 1×1 m quadrat: cover polygons filled; density as outlines (or filled).
    Axis is [0,1] in both directions with equal aspect.
    """
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)

    # Draw cover as filled polygons
    if len(cover_gdf):
        cover_gdf.plot(ax=ax, edgecolor='black', linewidth=0.5, alpha=0.7)  # default colormap by GeoPandas
    # Draw density polygons/points
    if len(density_gdf):
        # If density is points (rare here), GeoPandas will plot markers; if polygons, they’ll be tiny circles.
        density_gdf.plot(ax=ax, edgecolor='black', linewidth=0.2, alpha=0.9)

    # Limit to the 1×1 quadrat and make it square
    ax.set_xlim(0,1)
    ax.set_ylim(0,1)
    ax.set_aspect('equal', adjustable='box')

    # Simple legend substitute: annotate largest patches by species name (optional)
    try:
        # Label a few largest cover polygons to keep map clean
        if len(cover_gdf) and 'area' not in cover_gdf.columns:
            cover_gdf = cover_gdf.assign(area=cover_gdf.geometry.area)
        for _, row in cover_gdf.sort_values('area', ascending=False).head(5).iterrows():
            x, y = row.geometry.representative_point().coords[0]
            ax.text(x, y, str(row.get(label_field, ""))[:18], fontsize=6, ha='center', va='center')
    except Exception:
        pass

    ax.set_xticks([0,0.5,1.0]); ax.set_yticks([0,0.5,1.0])
    ax.grid(True, linewidth=0.2)
    if title:
        ax.set_title(title, fontsize=10)

    if out_png:
        fig.savefig(out_png, bbox_inches='tight')
    return fig, ax

# Example: make the 2002 map for quadrat 30711 to mirror your JPEG preview
root = "/path/to/unzipped/archive"
quad_id = 30711
year = 2002

cover_gdf, density_gdf = load_quadrat_layers(root, quad_id, year)
plot_quadrat(cover_gdf, density_gdf,
             title=f"Quadrat {quad_id} — {year}",
             out_png=f"quadrat_{quad_id}_{year}.png")